In [23]:
questions = ["I would like to upgrade the RideView apk installed in my dashcam. How do I do that ?",
"Can I request a video of 30 min duration?",
"What does LightMetrics do?",
"how do i disable End Trip On Idling ",
"What is a companion device?",
"I am getting lot of incorrect violations for speeding, what should I do",
"how can i mark events for coaching",
"can we customize role?",
"Who is Lisa",
"but where on the events modal can i find &#x27;Select for Coaching&#x27;",
"tell me about RideView pricing - how much does it cost?",
"Compare K245 and K265",
"What is the cost of one camera?",
"why is event video not uploaded",
"resolutions supported",
"How do u validate the AI models",
"What is the use if ADAS is disabled?",
"Tell me about the parking feature when the vehicle is parked",
"How to add a new fleet?",
"can we add image for external driver ?",
"vehicle speed is incorrect in the event video",
"can the device stop the car before collision?",
"Difference between JC400 and JC400p?",
"what is the latest APK version?",
"what are portal notifications",
"How do I connect a 3rd camera?",
"What is RideView?",
"How accurate are the AI models",
"What are the events available on kafka queue?",
"Which is better JC400 or K245?",
"which modules have different thresholds for heavy and light duty vehicles",
"What is the difference between JC400 and JC400P",
"If data connection to a. camera is lost during a trip, will the driver get a notification?",
"Is it possible to connect an additional camera apart from road and driver camera?",
"how does one disable DMS",
"what is trip mode?",
"What is a DVR",
"How many vehicles can a fleet contain",
"If a driver does not tap the NFC card at the start of a trip , can a notification be given to the driver?",
"how many cameras does mitac gemini support",
"How much time it takes to fulfill a DVR video request?",
"How to fix when the dash cam has sensor error?",
"difference between parking mode and surveillance mode",
"whats a traffic sign violation event",
"what are custom events",
"Give me an example of the status timeline ",
" am not getting audio alerts eventhough the events can be seen on the portal",
"How does provisioning work in RideView?",
"What is the use of companion App ?",
"What are the guidelines?",
"What is status timeline in a DVR video requests?",
"What are the different camera health events available on the portal?",
"Do the forward facing solutions work during rain",
"how to connect to lightmetrics kafka and consume messages",
"How many different types of report one gets with Rideview",
"I need to request a video ",
"How to allocate a dash to a different fleet?",
"How do I see how much data has been used by a fleet?",
"What is EVO?",
"how do I activate privacy mode",
"Which SD Cards are supported?",
"can you describe this in more detail",
"is there a companion app",
"Can I assign driver ID after the trip?",
"tell me something about rideview",
"how to use live telematics",
"How do I know if my camera is connected to network or not?",
"I want my fleet portal to have my company logo",
"how do I initiate live streaming from the portal",
"Where do i get comprehensive weekly report on my fleet",
"I am unable get any distraction events, what should I do",
"I have a missing trip, where can I find it in the portal",
"What are the different driver ID options available to associate a driver to a trip?",
"How to move a dash cam from one fleet to another?",
"can you give a sample curl request for listing violations",
"what is LOCEE. please explain",
"What is the services tab in the master portal used for",
"device is repeating audio alerts",
"Which ADAS events are supported?",
"Whats the latest version of rideview fleet portal?",
"How do you install a camera?",
"where is trip data stored ? ",
"My vehicle is stolen, can I stop it remotely?",
"as a fleet manager where can i check the performance of the drivers in the fleet, please provide the instructions to access the information in steps",
"What are the different kinds of reports available on RideView?",
"How to enable audio recording on the dash cam?",
"How do i install. rideview dashcam in my truck",
"Is it possible to switch off the audio on my camera",
"Can I listen to drivers from fleet portal?",
"how to integrate lightmetrics APIs",
"do you have repeated audio alerts on devices",
"what are the functionalities in latest release",
"Can I do rate limiting at a fleet level?",
"What are the different privileges available to a user in the Master Portal?",
"can I set different violation configuration for each asset",
"how will the driver side affect ADAS detections?",
"What event detections are supported for driver camera?",
"How to know if any of my trucks got into an accident?",
"How does rate limiting work?",
"How can fleet manager accept or reject a challenge raised by a driver?"]

In [11]:
import os
from dotenv import load_dotenv
from utils.vectorstore import new_vectorstore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.schema import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


load_dotenv()
llm = ChatOpenAI()

system_prompt = (
    "You are an AI assistant designed to answer questions using the provided context. Your response should strictly adhere to the context and be clear, concise, and structured. Please follow these guidelines:"

    "1. **Relevance**: Use only the provided context to answer the question. If the context does not contain enough information, respond: *'Unfortunately, I am unable to answer that question'* Do not guess or fabricate answers."

    "2. **Clarity and Completeness**:"
    "- Provide complete answers wherever possible."
    "- Break down complex responses into bullet points or numbered steps for better readability."

    "3. **Examples and Specificity**:"
    "- Include relevant examples or specific details from the context when applicable."
    "- If examples are not available in the context, avoid making them up."

    "4. **Alternative Suggestions**:"
    "- If the question cannot be fully answered, suggest steps the user could take (e.g., 'Refer to the platform's documentation' or 'Contact support for clarification')."

    "5. **Error Handling**:"
    "- If the question is ambiguous or lacks context, state that directly and request more details if appropriate."

    "6. **Structured Greetings and Acknowledgments**:"
    "- Respond to greetings, acknowledgments, and thank-you messages appropriately and politely."
    "question: {input}"
    "context: {context}"
)

qa_prompt = PromptTemplate.from_template(system_prompt)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [12]:
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import ContextualCompressionRetriever


vectorstore = new_vectorstore(None)

retriever = vectorstore.as_retriever(
        search_type="similarity"
    )

embeddings = OpenAIEmbeddings()
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever
)

/Users/hitheshbhat/Documents/engineering/chat-knowledge-base/utils/vectorstore.py:17: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  return PGVector(


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List



condense_question_system_template = (
    "Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language."
    "If the follow up question does not need context, return the exact same text back."
    "Never rephrase the follow up question given the chat history unless the follow up question needs context."
    "Do not answer the question"
    "Chat History: {chat_history}"
    "Follow Up question: {input}"
    "Standalone question:"
)
condense_question_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", condense_question_system_template),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm, compression_retriever, condense_question_prompt
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """In memory implementation of chat message history."""

    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """Add a list of messages to the store"""
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]


qa_chain = create_stuff_documents_chain(llm, qa_prompt)

convo_qa_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

chain_with_history = RunnableWithMessageHistory(
    convo_qa_chain,
    get_by_session_id,
    input_messages_key="input",
    history_messages_key="chat_history"
)

a = convo_qa_chain.invoke(
    {
        "input": "where is it's office?"
    }, config={"configurable": {"session_id": "abc123"}, "verbose": True}
)

a

